# Variational inference for Gaussian Mixture Model using approximate fractional posteriors

## Key parameters

In [ ]:
### Settings 1 for Table 1 in main paper
#### one of K or mu must be set
K = None            # number of components
mu = [-2, 2]     # [-2, 2 ] (Syring)
m_init = [-1, 1] # optional initisation. Can be None
sigma2 = 3*3     # prior variance of the component means. Analysis is not sensitive to this parameter. we currently set the sd to 1.5 of the true mu

n = 400          # Oberverations per data set
nreps = 5000     # Number of data sets for frequentist analysis
alpha = 0.05     # signifiance level for calibration checks
gammas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

#### Tolerances for algorithms
tol = 1e-5   # convergence tolerance
eps = 1e-20  # small probabilities are reduced to zeros

#### Seed for reproducibility
seed = 23

#### System
nprocesses = 6

In [ ]:
### Common Settings

#### Importance sampling
##### Settings for doing for all repetitions
isns = 1000 # number of samples for importance sampling
isidx = -1   # replicate from which we are getting the importance sampling result, so that we don't waste compute

##### Settings for doing for one repetition
#isns = 100000 # number of samples for importance sampling
#isidx = 0   # replicate from which we are getting the importance sampling result, so that we don't waste compute

#### Tolerances for algorithms
tol = 1e-5   # convergence tolerance
eps = 1e-20  # small probabilities are reduced to zeros

#### Seed for reproducibility
seed = 23

#### System
nprocesses = 6

## Setup

In [ ]:
import numpy as np
np.random.seed(seed)

### we are using normal distributions in this experiment
from scipy.stats import norm
critical_value = norm.ppf(1-alpha/2)

import platform
import multiprocessing
nprocesses_max = multiprocessing.cpu_count()
if platform.system() == "Darwin":
    multiprocessing.set_start_method('fork') # Need this to work on MacOS

if not mu is None and K is None:
    K = len(mu)

# based on CLT, white noise, uniform cluster assignment, we obtain a length that we are expecting
# There are n/K points per cluster
# The expected probability for each point can be set above 0.5
# So expected sample size is prob * n/K
expected_prob_per_point = 1.0 # depends on how well we believe the cluster is separated 
expected_length = 2 * critical_value / np.sqrt(expected_prob_per_point * n/K)

# We are mixing the ELBO means with the std from these
ref_gammas_idx = [0, 4, 8]

In [ ]:
### Logging and checking
if not mu is None and len(mu) != K:
    raise Error("number of components mismatch")

print("seed None" if seed is None else "Seed %d" % seed)
print("critical %.3f" % critical_value)
print("K %d" % K)
print("n/K %.3f" % (n/K))
print("R %d" % nreps)
print("length %.3f" % expected_length)
if mu is None:
    print("mu None")
else:
    print("mu", mu)

if m_init is None:
    print("m_init None")
else:
    print("m_init", m_init)

print("nprocesses %d/%d" % (nprocesses, multiprocessing.cpu_count()))
if( nprocesses >= nprocesses_max ):
    raise Error("Let's not hog the system")

## Algorithms

In [ ]:
from scipy.special import softmax

In [ ]:
def fix_small_probabilities(varphi, eps):
  varphi_eps = varphi < eps
  varphi[varphi_eps] = 0.0
  varphi = varphi / np.sum(varphi, axis=1, keepdims=True)
  return varphi, varphi_eps

In [ ]:
def GMMelbo(x, K, sigma2, eps, m, s2):
  """
  Cut and paste from GMMvarinf
  """
  n = len(x)
  x = np.expand_dims(x, axis=1) # n-by-1 matrix

  dist2 = (x - m) ** 2
  logvarphi = (-dist2 - s2)/2
  varphi = softmax(logvarphi, axis=1)

  ## Divergences, taking care of zero measures
  _varphi, _varphi_eps = fix_small_probabilities(varphi, eps)
  _varphi[_varphi_eps] = 1.0 # set to one because we are doing logarithm
  sDc = np.sum(_varphi * np.log(_varphi * K)) # uniform prior for assigment probability

  Dmu = (m*m + s2 - sigma2) / (2*sigma2) + np.log(sigma2 / s2)
  sDmu = np.sum(Dmu)

  ## Data fit
  _varphi[_varphi_eps] = 0.0 # set to zero to remove effects of bad logvarphi's
  sdata = np.sum(_varphi * logvarphi) - np.log(2*np.pi) * (n/2)

  bound = sdata - sDmu - sDc
  return bound

In [ ]:
def GMMvarinf(x, K, sigma2, tol, eps, m = None):
  n = len(x)
  x = np.expand_dims(x, axis=1) # n-by-1 matrix

  # initialize
  if m is None:
      m = np.random.normal(0, sigma2, (1, K)) # with prior
  s2 = np.full((1,K), n/K) # This is what we can obtain for uniform cluster assignments and unit variance likelihood

  prevbound = 0
  bound = 0
  iter = 0
  while (bound > prevbound + tol) or (iter < 3):
    dist2 = (x - m) ** 2
    logvarphi = (-dist2 - s2)/2
    varphi = softmax(logvarphi, axis=1)

    xvarphi = x.T @ varphi
    svarphi = np.sum(varphi, axis=0, keepdims = True)
    s2 = 1.0 / (1.0/sigma2 + svarphi)
    m = xvarphi * s2

    # Compute Objective
    # Our bound is only correct at convergence. We make use the fix-point property of the update

    ## Divergences, taking care of zero measures
    _varphi, _varphi_eps = fix_small_probabilities(varphi, eps)
    _varphi[_varphi_eps] = 1.0 # set to one because we are doing logarithm
    sDc = np.sum(_varphi * np.log(_varphi * K)) # uniform prior for assigment probability

    Dmu = (m*m + s2 - sigma2) / (2*sigma2) + np.log(sigma2 / s2)
    sDmu = np.sum(Dmu)

    ## Data fit
    _varphi[_varphi_eps] = 0.0 # set to zero to remove effects of bad logvarphi's
    sdata = np.sum(_varphi * logvarphi) - np.log(2*np.pi) * (n/2)

    # Progress
    prevbound = bound
    bound = sdata - sDmu - sDc
    iter = iter + 1
    #print( "%3d %.3f %.3f %.3f %.3f" % (iter, bound, sdata, sDmu, sDc))

  return m, s2, varphi, bound, sdata, sDmu, sDc

In [ ]:
from scipy.special import softmax

def GMMvarfracinf(x, K, sigma2, gamma, tol, eps, m = None):
  alpha = 1/gamma
  n = len(x)
  x = np.expand_dims(x, axis=1) # n-by-1 matrix

  # initialize
  if m is None:
      m = np.random.normal(0, sigma2, (1, K)) # with prior
  s2 = np.full((1,K), n/K) # This is what we can obtain for uniform cluster assignments and unit variance likelihood

  # Start varphi with the ELBO initilisation
  logvarphi = (-(x - m) ** 2 - s2)/2
  varphi = softmax(logvarphi, axis=1)

  prevbound = 0
  bound = 0
  iter = 0
  while (bound > prevbound + tol) or (iter < 3):
    # re-introduce selected omitted data
    s2i = 1.0 / (1.0/ s2 + (1.0-gamma) * varphi)
    mi = (m / s2 + (1.0-gamma) * varphi * x) * s2i

    dist2i = (x - mi) ** 2
    logvarphi = (-dist2i - s2i)/2
    varphi = softmax(logvarphi, axis=1)

    xvarphi = x.T @ varphi
    svarphi = np.sum(varphi, axis=0, keepdims=True)
    s2 = 1.0 / (1.0/sigma2 + gamma * svarphi)
    m = gamma * xvarphi * s2

    # Compute Objective
    # Our bound is only correct at convergence. We make use the fix-point property of the update

    ## Divergences, taking care of zero measures
    _varphi, _varphi_eps = fix_small_probabilities(varphi, eps)
    _varphi[_varphi_eps] = 1.0 # set to one because we are doing logarithm
    sDc = np.sum(_varphi * np.log(_varphi * K)) # uniform prior for assigment probability

    s2a = alpha * sigma2 + (1-alpha) * s2
    Dmu =  (m*m / s2a) * (alpha/2) + np.log(sigma2/s2a) / (2*(alpha-1)) + np.log(sigma2 / s2)
    sDmu = np.sum(Dmu)

    ## Data fit
    #_varphi[_varphi_eps] = 0.0 # set to zero because we are mulitplying
    s2varphi1gamma1 = s2 * varphi * (1.0-gamma) + 1.0
    dist2 = (x - m) ** 2
    sdata = 0.5 * ( -np.sum(np.log(s2varphi1gamma1)) / (1-gamma) - np.sum(varphi * dist2 / s2varphi1gamma1) - n * np.log(2*np.pi))

    # Progress
    prevbound = bound
    bound = sdata - sDmu - sDc
    iter = iter + 1
    #print( "%3d %.3f %.3f %.3f %.3f" % (iter, bound, sdata, sDmu, sDc))

  return m, s2, varphi, bound, sdata, sDmu, sDc

In [ ]:
from scipy.stats import norm
from scipy.special import logsumexp

def importance_sampler_for_evd(ns, sigma2, x, m, s2, varphi):
    """
    ns: number of importance samples (and hence weights)
    sigma2: prior variance of componment means (centered at zero)
    x: data
    m, s2: approximate posterior mean and variances
    varphi: data->cluster assignment
    """
    n = len(x)
    K = m.shape[1]

    # Generate samples
    mu = np.random.normal(m, np.sqrt(s2), (ns, K))
    cumvarphi = np.expand_dims(np.cumsum(varphi, axis=1), axis=0)
    u = np.random.uniform(size=[ns, n, 1])
    c = np.cumsum(u > cumvarphi, axis=2) == np.arange(0,K)[np.newaxis, np.newaxis, :]
    muc = np.repeat(np.expand_dims(mu, axis=1), n, axis=1)[c]
    muc = np.reshape(muc, (ns, -1))
    
    # Compute likelihoods
    x = np.expand_dims(x, axis=0)
    llh = norm.logpdf(x, muc, 1)

    # Compute priors and posteriors of components
    mu_logprior = norm.logpdf(mu, 0, np.sqrt(sigma2))
    mu_logposterior = norm.logpdf(mu, m, np.sqrt(s2))

    # Compute posterior of component
    c_prior = 1.0/K
    c_posterior = np.repeat(np.expand_dims(varphi, axis=0), ns, axis=0)[c]
    c_posterior = np.reshape(c_posterior, (ns, -1))

    # Weight of samples
    logw = np.sum(llh + np.log(c_prior) - np.log(c_posterior), axis=1) + np.sum(mu_logprior - mu_logposterior, axis=1)
    log_meanw = logsumexp(logw) - np.log(ns)
    cv = np.sqrt( np.mean( (np.exp(logw - log_meanw) - 1) ** 2) )
    
    #print("shapes ", c.shape, mu.shape, muc.shape, x.shape, llh.shape, mu_logprior.shape, mu_logposterior.shape, c_posterior.shape, logw.shape) # DEBUG
    return log_meanw, cv

## Calibration tools

In [ ]:
from scipy.stats import linregress

In [ ]:
def generate_observations(n, K, sigma2, nreps, mu = None):
  if mu is None:
      mu = np.random.normal(0, np.sqrt(sigma2), K)
  else:
      mu = np.asarray(mu)
  c = np.random.choice(range(K), n, p=[1.0/K]*K)
  muc = mu[c]
  x = np.random.normal(muc, 1, (nreps, n));
  # print("shapes ", c.shape, mu.shape, muc.shape, x.shape) # DEBUG    
  return x, mu, c

# We need to order the means so that comparision between sets of means make sense
def sort_components(m, s2):
  i = np.argsort(m, axis=1)
  return m[0, i[0]], s2[0, i[0]]

In [ ]:
def calibration_diagnostics(mu, m, s2, critical_value):
  cs = critical_value * np.sqrt(s2)
  lb = m - cs
  ub = m + cs
  cover = ((mu >= lb) & (mu <= ub)).astype(int)
  return cover, 2 * cs

In [ ]:
def do_replication_gamma(gamma, x, dois = False):
  m, s2, varphi, bound, Ddata, Dmu, Dc = GMMvarfracinf(x, K, sigma2, gamma, tol, eps, m_init) ## DEBUG
  if dois:
      logevd, cv = importance_sampler_for_evd(isns, sigma2, x, m, s2, varphi)    
  else:
      logevd, cv = 0, 0
      
  m, s2 = sort_components(m, s2)  # beware, we did not sort varphi
  cover, length = calibration_diagnostics(mu, m, s2, critical_value)
  return cover, length, bound, m, s2, logevd, cv

In [ ]:
def do_replication(xi):
  x, repidx = xi
    
  ngammas = len(gammas)  
  nrefgammas = len(ref_gammas_idx)
  nalgos = ngammas + 1 + nrefgammas + 2 # add ELBO and predictions based on interval length
    
  ms = np.zeros( (nalgos, K) )
  s2s = np.zeros( (nalgos, K) )    
  covers = np.zeros( (nalgos, K) )
  lengths = np.zeros( (nalgos, K) )
  bounds = np.zeros( (nalgos) )
  logevds = np.zeros( (nalgos) )
  cvs = np.zeros( (nalgos) )    
  
  for g in range(ngammas):
    """    
    m, s2, varphi, bound, Ddata, Dmu, Dc = GMMvarfracinf(x, K, sigma2, gammas[g], tol, eps, m_init) ## DEBUG
    m, s2 = sort_components(m, s2)  # beware, we did not sort varphi
    ms[g, :], s2s[g,:], bounds[g] = m, s2, bound
    covers[g, :], lengths[g, :] = calibration_diagnostics(mu, m, s2, critical_value)
    """
    covers[g, :], lengths[g, :], bounds[g], ms[g, :], s2s[g,:], logevds[g], cvs[g] = do_replication_gamma(gammas[g], x, repidx == isidx or isidx == -1)
    
  # ELBO
  elbo = ngammas
  g = elbo
  m, s2, varphi, bound, Ddata, Dmu, Dc = GMMvarinf(x, K, sigma2, tol, eps, m_init)
  if repidx == isidx or isidx == -1:
     logevds[g], cvs[g] = importance_sampler_for_evd(isns, sigma2, x, m, s2, varphi)        
  m, s2 = sort_components(m, s2)  # beware, we did not sort varphi
  ms[g, :], s2s[g,:], bounds[g] = m, s2, bound
  covers[g, :], lengths[g, :] = calibration_diagnostics(mu, m, s2, critical_value)
    
  # Just expand the variance and check if calibration is better
  for r in range(nrefgammas):
      refg = ref_gammas_idx[r]
      g = elbo + r + 1
      m, s2 = ms[elbo, :], s2s[refg, :]
      bound = GMMelbo(x, K, sigma2, eps, m, s2)
      ms[g, :], s2s[g,:], bounds[g] = m, s2, bound
      covers[g, :], lengths[g, :] = calibration_diagnostics(mu, m, s2, critical_value)

  # linear regression to target the interval length
  predg = 0.0
  for k in range(K):
      res = linregress(lengths[0:(elbo+1), k], np.append(gammas, 1.0))
      predg = predg + np.clip( res.intercept + res.slope * expected_length, 0.0, 1.0)
  predg1 = predg / K

  g = elbo + nrefgammas + 1
  m, s2, varphi, bound, Ddata, Dmu, Dc = GMMvarfracinf(x, K, sigma2, predg1, tol, eps, m_init) ## DEBUG
  m, s2 = sort_components(m, s2)  # beware, we did not sort varphi
  ms[g, :], s2s[g,:], bounds[g] = m, s2, bound
  covers[g, :], lengths[g, :] = calibration_diagnostics(mu, m, s2, critical_value)    
    
  # linear regression to target the inverse sqauired interval length (which makes sense from the mixing point of view
  predg = 0.0
  for k in range(K):
      res = linregress(1.0/(lengths[0:(elbo+1), k] ** 2), np.append(gammas, 1.0))
      predg = predg + np.clip(res.intercept + res.slope / (expected_length ** 2), 0.0, 1.0)
  predg2 = predg / K

  g = elbo + nrefgammas + 2
  m, s2, varphi, bound, Ddata, Dmu, Dc = GMMvarfracinf(x, K, sigma2, predg2, tol, eps, m_init) ## DEBUG
  m, s2 = sort_components(m, s2)  # beware, we did not sort varphi
  ms[g, :], s2s[g,:], bounds[g] = m, s2, bound
  covers[g, :], lengths[g, :] = calibration_diagnostics(mu, m, s2, critical_value)    

  return covers, lengths, bounds, predg1, predg2, logevds, cvs

## Experiment

In [ ]:
def add_tuples(t1, t2):
    if len(t1) == 0:
        return ()
    else:
        return (t1[0] + t2[0],) + add_tuples(t1[1:], t2[1:])

In [ ]:
data, mu, _ = generate_observations(n, K, sigma2, nreps, mu)

if m_init is None:
    m_init = mu  + np.random.normal(0, 1, K);# everyone is initialised the same

# sort for diagnoistics later
mu = np.sort(mu)  
m_init = np.sort(m_init)

with multiprocessing.Pool(processes=nprocesses) as pool:
    results = pool.map(do_replication, [ (data[i,:], i) for i in range(nreps)])

In [ ]:
from functools import reduce

r = reduce(lambda x, y: add_tuples(x,y), results)
r = [x/nreps for x in r]

np.set_printoptions(precision = 5)
print(mu)
print(m_init)
print(expected_length)
print(r[0])
print(r[1])
print(r[2])
print(r[3])

In [ ]:
# regress to target coverage
predg = 0.0
m = len(gammas)+1
covers = r[0]
for k in range(K):
    res = linregress(covers[0:m, k], np.append(gammas, 1.0))
    predg = predg + (res.intercept + res.slope * (1-alpha))  
predg = predg / K


In [ ]:
def do_replication_gamma_fixed(x):
    return do_replication_gamma(predg, x)[:3]

In [ ]:
with multiprocessing.Pool(processes=nprocesses) as pool:
    results_last = pool.map(do_replication_gamma_fixed, [data[i,:] for i in range(nreps)])

In [ ]:
r_last = reduce(lambda x, y: add_tuples(x,y), results_last)
r_last = [x/nreps for x in r_last]
print(r_last)

In [ ]:
### pretty print for LaTeX
results_str = " & $%.4f$ & $%.4f$" * K
for i in range(len(gammas)):
    print( ("$\\LB_{%.1f}$" + results_str + " & $%.1f$ \\\\") % (tuple([gammas[i]]) + tuple( [x for xs in [ [r[0][i,k], r[1][i,k]] for k in range(K)] for x in xs] ) + tuple([r[2][i]])))

i = len(gammas)
print( ("$\\LB_{%.1f}$" + results_str + " & $%.1f$ \\\\") % (tuple([1.0]) + tuple( [x for xs in [ [r[0][i,k], r[1][i,k]] for k in range(K)] for x in xs] ) + tuple([r[2][i]])))

for ii in range(len(ref_gammas_idx)):
    i = len(gammas)+1+ii
    print( ("$C_{%.1f}$" + results_str + " & $%.1f$ \\\\") % (tuple([gammas[ref_gammas_idx[ii]]]) + tuple( [x for xs in [ [r[0][i,k], r[1][i,k]] for k in range(K)] for x in xs] ) + tuple([r[2][i]])))
    
i = len(gammas) + 1 + len(ref_gammas_idx)
print( ("$R_\\ell$" + results_str + " & $%.1f$ \\\\") % (tuple( [x for xs in [ [r[0][i,k], r[1][i,k]] for k in range(K)] for x in xs] ) + tuple([r[2][i]])))

i = len(gammas) + 1 + len(ref_gammas_idx) + 1
print( ("$R_{\\ell^{-2}}$" + results_str + " & $%.1f$ \\\\") % (tuple( [x for xs in [ [r[0][i,k], r[1][i,k]] for k in range(K)] for x in xs] ) + tuple([r[2][i]])))

print( ("$R_\\kappa$" + results_str + " & $%.1f$ \\\\") % (tuple( [x for xs in [ [r_last[0][k], r_last[1][k]] for k in range(K)] for x in xs] ) + tuple([r_last[2]])))

In [ ]:
print("expected_length = %0.3f, gamma(Rell) = %0.3f, gamma(Rell-2) = %0.3f, gamma(Rkappa) = %0.3f" % (expected_length, r[3], r[4], predg))

## Show non-linearity in interpolated precision

In [ ]:
import matplotlib.pyplot as plt

gamma1 = np.expand_dims(np.concatenate([gammas, [1.0]]), axis=0)

for k in range(K):
    precisions = np.stack([ 1./((r[1][:10, k]/2/critical_value)**2) for r in results])
    idealprec = (1-gamma1)/ sigma2 + np.expand_dims(precisions[:,-1], axis=1) * gamma1
    residue = (precisions - idealprec)**2
    residue = np.sum(residue[:, :-1], axis=1) / len(gammas) # Don't count the Bayes
    log10residue = np.log10(residue)
    print(residue.shape)
    bad = np.argmax(residue)
    print(K, bad, residue[bad], np.mean(residue), np.std(residue))
    print("bad example = ", precisions[bad,:])
    print("log residues = ", log10residue)
    fig, axs = plt.subplots(1, 2)
    badplot = np.concatenate([np.expand_dims(np.array([0.0, 1/sigma2, 1/sigma2]), axis=1), np.stack([gamma1[0,:], precisions[bad,:], idealprec[bad,:]])], axis=1)
    axs[0].plot(badplot[0,:], badplot[1,:])
    axs[0].plot(badplot[0,:], badplot[2,:])
    axs[1].hist(log10residue)
    plt.show()
    np.savetxt("results/badresidue-%d.txt" %k, np.transpose(badplot))
    np.savetxt("results/logresidue-%d.txt" % k, log10residue)

In [ ]:
# Importance sampling outcome
for i in range(len(gammas)):
    print("%.1f & %.1f & %.1f & %.1f" % (gammas[i], results[isidx][2][i], results[isidx][5][i], results[isidx][6][i]))
i=len(gammas)
print("%.1f & %.1f & %.1f & %.1f" % (1.0, results[isidx][2][i], results[isidx][5][i], results[isidx][6][i]))

In [ ]:
if isidx == -1:
    print(r[5][0:(len(gammas)+1)], '\n', r[6][0:(len(gammas)+1)]) # only make sense if isidx == -1